<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/17%20-%20Problemas%20Hamiltonianos%20e%20Roteamento%20de%20AGV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/17%20-%20Problemas%20Hamiltonianos%20e%20Roteamento%20de%20AGV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 17 - Notebook: Problema do Caixeiro-Viajante (TSP) e Roteamento Logístico de AGVs na Linha de Envase

Neste notebook implementamos a heurística do **Vizinho Mais Próximo** e a busca local **2-Opt** para otimizar o percurso fechado de amostragem e inspeção do AGV industrial na **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**.

In [ ]:
from typing import List, Dict, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[float]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "INF" if v == float('inf') else f"{v:.1f}"
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class RoteadorAGV_TSP:
    def __init__(self, locais: List[str], matriz_distancias: List[List[float]]):
        self.locais = locais
        self.dist = matriz_distancias
        self.n = len(locais)

    def calcular_custo_rota(self, rota: List[int]) -> float:
        """Calcula o custo total acumulado do ciclo fechado (retornando a origem)."""
        c = sum(self.dist[rota[i]][rota[i+1]] for i in range(len(rota)-1))
        c += self.dist[rota[-1]][rota[0]]
        return c

    def vizinho_mais_proximo(self, inicio_idx: int = 0) -> List[int]:
        """Heurística construtiva gulosa (Nearest Neighbor)."""
        nao_vis = set(range(self.n))
        nao_vis.remove(inicio_idx)
        rota = [inicio_idx]
        atual = inicio_idx
        while nao_vis:
            prox = min(nao_vis, key=lambda v: self.dist[atual][v])
            rota.append(prox)
            nao_vis.remove(prox)
            atual = prox
        return rota

    def otimizar_2opt(self, rota: List[int]) -> List[int]:
        """Refinamento por busca local 2-Opt para eliminação de cruzamentos."""
        melhor = list(rota)
        melhor_c = self.calcular_custo_rota(melhor)
        melhorou = True
        while melhorou:
            melhorou = False
            for i in range(1, self.n - 1):
                for j in range(i + 1, self.n):
                    nova = melhor[:i] + melhor[i:j+1][::-1] + melhor[j+1:]
                    c = self.calcular_custo_rota(nova)
                    if c < melhor_c:
                        melhor_c = c
                        melhor = nova
                        melhorou = True
                        break
                if melhorou:
                    break
        return melhor

# Estações de paragem do AGV na Célula de Envase (Grupo 3)
locais_amostragem = [
    "Lab_CQ",           # Base de recarga e bancada de análises
    "TS1_Suprimento",   # Ponto de colheita no tanque principal (Brix/pH)
    "BC1_Bomba",        # Monitorização de vibração e rolamentos do skid
    "AS1_Acumulador",   # Inspeção do vaso de pressão
    "VS2_Envase",       # Coletor de dosagem de envase
    "EST_Envase"        # Amostragem de garrafas no final da esteira
]

# Matriz métrica de distâncias de circulação industrial no piso da fábrica (m)
dist_matriz = [
    [0.0,  45.0, 50.0, 80.0, 75.0, 110.0],  # Lab_CQ
    [45.0,  0.0, 20.0, 55.0, 60.0,  95.0],  # TS1_Suprimento
    [50.0, 20.0,  0.0, 45.0, 50.0,  85.0],  # BC1_Bomba
    [80.0, 55.0, 45.0,  0.0, 25.0,  40.0],  # AS1_Acumulador
    [75.0, 60.0, 50.0, 25.0,  0.0,  35.0],  # VS2_Envase
    [110.0, 95.0, 85.0, 40.0, 35.0,   0.0]   # EST_Envase
]

print("=== MATRIZ DE DISTÂNCIAS ENTRE PONTOS DE AMOSTRAGEM DO AGV (METROS) ===")
print(formatar_matriz(dist_matriz, locais_amostragem, locais_amostragem))

agv_roteador = RoteadorAGV_TSP(locais_amostragem, dist_matriz)
r_nn = agv_roteador.vizinho_mais_proximo(0)
custo_nn = agv_roteador.calcular_custo_rota(r_nn)

r_opt = agv_roteador.otimizar_2opt(r_nn)
custo_opt = agv_roteador.calcular_custo_rota(r_opt)

# Formatação comparativa de resultados em ASCII
resumo_rotas = [
    {
        "Método": "Vizinho Mais Próximo (NN)",
        "Sequência Completa (Retorno à Base)": " -> ".join([locais_amostragem[i] for i in r_nn] + [locais_amostragem[r_nn[0]]]),
        "Distância Total (m)": f"{custo_nn:.1f}",
        "Eficiência": "Base Gulosa"
    },
    {
        "Método": "Refinamento 2-Opt",
        "Sequência Completa (Retorno à Base)": " -> ".join([locais_amostragem[i] for i in r_opt] + [locais_amostragem[r_opt[0]]]),
        "Distância Total (m)": f"{custo_opt:.1f}",
        "Eficiência": f"Redução de {custo_nn - custo_opt:.1f} m ({((custo_nn - custo_opt)/custo_nn)*100:.1f}%)"
    }
]

print("\n=== COMPARAÇÃO DE DESEMPENHO: ROTEAMENTO TSP DO AGV ===")
print(formatar_tabela(resumo_rotas))

# Asserções formais de integridade matemática
assert len(r_nn) == len(locais_amostragem), "A rota inicial deve conter todos os nós."
assert len(set(r_nn)) == len(locais_amostragem), "Nenhum nó intermediário pode ser repetido."
assert custo_opt < custo_nn, "O refinamento 2-Opt deve produzir uma rota estritamente menor que a inicial."
assert custo_opt == 260.0, f"Custo esperado de 260.0m, obtido: {custo_opt}"
print("\n✓ Validação formal concluída: Rota 2-Opt converge para o ótimo com eliminação de saltos redundantes!")